In [18]:
using DifferentialEquations
using Plots
include("Chemkin.jl")


species_names = ["N2", "O2", "CH4", "H2O", "CO2"]
n_s = 5

#temperature (assumed constant)
T = 1200 #K
P = 10e5 #Pa

R = 8.314 # J/molK

c_tot = P / (R*T) # ideal gas law

X0 = zeros(length(species_names)+1)
X0[1] = T # K
X0[2:end] = c_tot*[7.57/10.57, 2/10.57, 1/10.57, 0, 0] #mol/m^3, adiabatic mixture

# Stoichiometry: 2 O2 + CH4 -> 2 H2O + CO2
S = [0, -2, -1, 2, 1]  # N2 does not participate in the reaction
n_r = 1

species_dict = load_chemical_data("CHEMKIN-THERMDAT.txt")

Dict{String, Tuple{Float64, Vector{Float64}}} with 550 entries:
  "HSIC"          => (1500.0, [5.84954, 0.000762835, -9.97413e-8, -3.81159e-11,…
  "H2S"           => (1000.0, [2.88315, 0.00382783, -1.4234e-6, 2.498e-10, -1.6…
  "SIF3NHSIH3"    => (1000.0, [16.6994, 0.00778978, -8.11057e-7, -7.6502e-10, 1…
  "CLSI(CH3)2CH2" => (1500.0, [21.151, 0.00801827, -7.92425e-7, -3.29505e-10, 5…
  "GEF2"          => (1000.0, [4.76795, 0.00841094, -1.67642e-5, 1.56225e-8, -5…
  "CSICL3"        => (1500.0, [12.5054, 0.000533922, -2.58861e-7, 6.07531e-11, …
  "H2GAME"        => (600.0, [5.8316, 0.0122287, 3.03367e-7, -3.95694e-9, 1.225…
  "O2-"           => (1000.0, [3.88301, 0.000740787, -2.96178e-7, 5.7243e-11, -…
  "ASALME"        => (600.0, [7.12711, 0.00735786, 2.3008e-8, -2.2264e-9, 6.927…
  "CH2CLCHCL2"    => (1500.0, [16.1874, 0.00304768, -5.0115e-7, -1.5967e-11, 7.…
  "CHCLCCLOH"     => (1500.0, [14.1221, 0.00258376, -4.5769e-7, 5.21568e-12, 3.…
  "H2SI(CH3)CH2"  => (1500.0, [13.8883, 0.007

In [19]:
function Arrhenius(Y)
    T = Y[1]
    X = Y[2:end] * 1e-6 # Arrhenius takes /cm3
    # Constants from Westbrook & Dryer 1981
    n_CH4 = -0.3
    n_O2 = 1.3
    A = 1.3e8 
    Ea = 48.4e3 # cal/mol
    R = 1.987 # cal/mol/K
    
    # Ensure positive concentrations for exponentiation
    CH4_concentration = max(X[3], 0)
    O2_concentration = max(X[2], 0)
    
    k = A * exp(-Ea / (R * T)) # rate constant
    r = k * CH4_concentration^n_CH4 * O2_concentration^n_O2 # reaction rate
    return r * 1e6 # Back to /m3
end

Arrhenius (generic function with 1 method)

In [15]:
# Temperature rate of change due to reaction enthalpy
function dT(X, r)
    h0_vec = []
    cp_vec = []
    dH = sum( S .* [h0(X[1],species_dict[species_names[i]]) for i in 1:n_s] .* r)
    c_p = sum(X[2:end] .* [species_cp(X[1],species_dict[species_names[i]]) for i in 1:n_s])
    dT = -dH/c_p #J/mols * molK/J
    return dT[1]
end


dT (generic function with 1 method)

In [16]:
# Derivative function for the ODE system
function f!(dX, X, p, t)
    r = Arrhenius(X)
    dX[1] = dT(X,r)
    dX[2:end] .= r .* S  # Species concentrations change
end

# define timespan
tend = .1
tspan = (0, tend)

# define problem 
problem = ODEProblem(f!, X0, tspan)

# solve problem 
@time sol = solve(problem, alg_hints=[:stiff], abstol = 1e-6, reltol = 1e-4)

retcode: Success
Interpolation: 3rd order Hermite
t: 1093-element Vector{Float64}:
 0.0
 0.0019262574212853504
 0.007676675016891531
 0.01666697162756464
 0.02735835909339371
 0.04013450882985927
 0.053892984352793466
 0.05985073468970777
 0.0667090758821712
 0.06893957726219963
 0.0696913480002758
 0.07082334060494647
 0.07136885267101228
 ⋮
 0.07710123306195948
 0.07710333593685036
 0.07710754168663211
 0.07711595318619562
 0.07711931778602102
 0.07712604698567183
 0.07712873866553215
 0.07713412202525279
 0.07718795562245923
 0.07772629159452359
 0.08310965131516722
 0.1
u: 1093-element Vector{Vector{Float64}}:
 [1200.0, 71.78432582323252, 18.96547577892537, 9.482737889462685, 0.0, 0.0]
 [1201.894871420468, 71.78432582323252, 18.947315819195563, 9.473657909597781, 0.018159959729808373, 0.009079979864904187]
 [1207.923567697446, 71.78432582323252, 18.889503345004485, 9.444751672502242, 0.0759724339208871, 0.03798621696044355]
 [1218.6950746338348, 71.78432582323252, 18.78607782860904

  0.653990 seconds (1.64 M allocations: 61.179 MiB, 81.24% compilation time: 17% of which was recompilation)


In [17]:
# Extract the solution arrays
t = sol.t
T = sol[1, :]
C = sol[2:end, :]'

println(maximum(T))

# Create a plot for the temperature
plot1 = plot(t, T, xlabel="Time (s)", ylabel="Temperature (K)")

# Create a plot for the species concentrations
plot2 = plot(t, C, xlabel="Time (s)", ylabel="Concentration", label=["N2" "O2" "CH4" "H2O" "CO2"])

plot(plot1, plot2, layout = (2,1))

savefig("1stepthermo")

3044.237336789115


"C:\\Users\\jelte\\chemicalcombustion\\Toy Models\\1stepthermo.png"

In [6]:
using LinearAlgebra: norm

function compute_relaxation_time(sol; tol=1e-6)
    """
    Compute relaxation time τ for a solution `sol` of an ODEProblem.
    
    τ is defined as the time when ‖u(t) - u_steady‖ / ‖u0 - u_steady‖ ≈ 1/e.
    
    Args:
        sol: Solution object from DifferentialEquations.jl
        tol: Tolerance for checking convergence (default: 1e-6)
    
    Returns:
        τ: Relaxation time (first time when decay reaches 1/e)
        If no such time is found, returns `nothing`.
    """
    u0 = sol.prob.u0          # Initial (perturbed) state
    u_steady = sol[end]       # Equilibrium state (final value)
    Δ0 = norm(u0 - u_steady)  # Initial deviation magnitude
    
    # Handle cases where Δ0 ≈ 0 (no perturbation)
    if Δ0 < tol
        @warn "Initial state is already at equilibrium (‖Δu‖ = $Δ0). τ is undefined."
        return nothing
    end
    
    # Target deviation: Δ0 / e
    target_deviation = Δ0 / MathConstants.e
    
    # Find the first time when ‖u(t) - u_steady‖ ≤ target_deviation
    τ = nothing
    for (i, t) in enumerate(sol.t)
        Δu = norm(sol.u[i] - u_steady)
        if Δu ≤ target_deviation + tol  # Allow numerical tolerance
            τ = t
            break
        end
    end
    
    if τ === nothing
        @warn "No relaxation time found within solution timeframe. Increase `tspan`."
    end
    
    return τ
end

compute_relaxation_time(sol)

0.07188571801094705